In [ ]:
# ============================================================
# BIASCOPE — Notebook 08a (v2): Calibration (temperature scaling) — CPU only
# FIX: val me MBIB ke English+Hindi rows same row_id share karte hain => row-wise dedupe nahi
# Inputs: outputs/nb07b_babe/predictions.csv (+ predictions_07c.csv)
# ============================================================
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.optimize import minimize_scalar
from sklearn.metrics import roc_auc_score
from google.colab import drive

drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/BIASCOPE")
SRC = BASE_DIR / "outputs" / "nb07b_babe"
OUT = BASE_DIR / "outputs" / "nb08_calibration"
OUT.mkdir(parents=True, exist_ok=True)
EPS = 1e-6
FINAL_RUN = "xlmr|lora|mbib+babe|s42"

# ---------------- load predictions (+ meta) ----------------
p07b = pd.read_csv(SRC / "predictions.csv")
r07b = pd.read_csv(SRC / "results.csv")
if (SRC / "predictions_07c.csv").exists():
    p07c = pd.read_csv(SRC / "predictions_07c.csv")
    r07c = pd.read_csv(SRC / "results_07c.csv")
    p07b = p07b[~p07b["run_id"].isin(set(p07c["run_id"]))]     # jo run 07c me hai uske 07b rows poore hata do
    P = pd.concat([p07b, p07c], ignore_index=True)
    metas = [r07b, r07c]
    print("07c predictions mili — same run_id par unhe prefer kiya gaya.")
else:
    P, metas = p07b, [r07b]
    print("⚠ 07c predictions nahi mili — sirf 07b (seed 42) ke runs use honge.")
meta = pd.concat(metas).drop_duplicates("run_id", keep="last")[["run_id", "model", "method", "train_set", "seed"]]
P = P.merge(meta, on="run_id")
print("Runs:", P["run_id"].nunique(), "| eval sets:", sorted(P["eval_set"].unique()))
print("val rows per run:", P[P["eval_set"] == "val"].groupby("run_id").size().to_dict())

# ---------------- helpers ----------------
def logit(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def nll(y, p):
    p = np.clip(p, EPS, 1 - EPS)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

def brier(y, p):
    return float(np.mean((p - y) ** 2))

def ece(y, p, bins=15):
    conf = np.maximum(p, 1 - p)
    correct = ((p >= 0.5).astype(int) == y).astype(float)
    edges = np.linspace(0.5, 1.0, bins + 1)
    edges[-1] = 1.0001
    e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf >= lo) & (conf < hi)
        if m.any():
            e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)

def fit_T(y, p):
    z = logit(p)
    res = minimize_scalar(lambda lt: nll(y, sigmoid(z / np.exp(lt))), bounds=(-3, 3), method="bounded")
    return float(np.exp(res.x))

# ---------------- fit T on val, evaluate on every eval set ----------------
EVALS = [e for e in ["mbib_en_test", "mbib_hi_test", "babe_test_en", "babe_test_hi", "gold_conf2"]
         if e in set(P["eval_set"])]
rows, temps = [], {}
for run_id, g in P.groupby("run_id"):
    val = g[g["eval_set"] == "val"]
    if val.empty:
        continue
    T = fit_T(val["y"].values, val["p"].values)
    temps[run_id] = T
    mrow = meta[meta["run_id"] == run_id].iloc[0]
    for ev in EVALS:
        d = g[g["eval_set"] == ev]
        if d.empty:
            continue
        y, p = d["y"].values.astype(int), d["p"].values
        pT = sigmoid(logit(p) / T)
        rows.append(dict(run_id=run_id, model=mrow["model"], method=mrow["method"],
                         train_set=mrow["train_set"], seed=mrow["seed"], T=T, eval_set=ev, n=len(d),
                         ece_before=ece(y, p), ece_after=ece(y, pT),
                         brier_before=brier(y, p), brier_after=brier(y, pT),
                         nll_before=nll(y, p), nll_after=nll(y, pT),
                         auc=roc_auc_score(y, p) if len(set(y)) == 2 else np.nan))
C = pd.DataFrame(rows)
C.to_csv(OUT / "calibration_results.csv", index=False)
json.dump(temps, open(OUT / "temperatures.json", "w"), indent=2)

if FINAL_RUN in temps:
    fd = BASE_DIR / "models" / "final"
    fd.mkdir(parents=True, exist_ok=True)
    json.dump({"run_id": FINAL_RUN, "temperature": temps[FINAL_RUN],
               "note": "fit on full mbib+babe validation set (no row_id dedupe); calibrated p = sigmoid(logit(p)/T)"},
              open(fd / "final_temperature.json", "w"), indent=2)
    print(f"\nFinal temperature saved: T={temps[FINAL_RUN]:.4f} -> {fd / 'final_temperature.json'}")
else:
    print(f"\n⚠ {FINAL_RUN} ke liye temperature nahi mili.")

# ---------------- tables ----------------
print("\n" + "=" * 70 + "\nTEMPERATURES (T>1 => model overconfident tha)\n" + "=" * 70)
display(C.drop_duplicates("run_id")[["run_id", "T"]].set_index("run_id").round(4))

def before_after(metric):
    t = C.copy()
    t["cell"] = t.apply(lambda r: f"{r[metric + '_before']:.3f} → {r[metric + '_after']:.3f}", axis=1)
    return t.pivot_table(index=["model", "method", "train_set", "seed"], columns="eval_set",
                         values="cell", aggfunc="first")[EVALS]

for metric in ["ece", "nll"]:
    print(f"\n--- {metric.upper()} before → after temperature scaling ---")
    display(before_after(metric))

print("\n--- Mean ECE over all runs, per eval set ---")
display(C.groupby("eval_set")[["ece_before", "ece_after"]].mean().loc[EVALS].round(3))
print("\n--- Mean ECE by method (LoRA vs Full), before → after ---")
display(C.groupby(["method", "eval_set"])[["ece_before", "ece_after"]].mean().round(3))

fin = C[C["train_set"] == "mbib+babe"]
if fin["seed"].nunique() > 1:
    def ms(x):
        return f"{x.mean():.3f}±{x.std():.3f}" if len(x) > 1 else f"{x.mean():.3f}"
    print("\n--- Final config (mbib+babe): ECE before / after, mean±std over seeds ---")
    display(fin.groupby(["model", "method", "eval_set"])[["ece_before", "ece_after"]].agg(ms))
    print("Seeds per config:", fin.groupby(["model", "method"])["seed"].nunique().to_dict())

# ---------------- reliability diagrams (final config) ----------------
final = {"full": "xlmr|full|mbib+babe|s42", "lora": FINAL_RUN}
show_sets = [e for e in ["babe_test_en", "gold_conf2"] if e in EVALS]
if all(r in temps for r in final.values()) and show_sets:
    fig, axes = plt.subplots(len(show_sets), 2, figsize=(10, 4.2 * len(show_sets)), squeeze=False)
    for j, (meth, rid) in enumerate(final.items()):
        for i, ev in enumerate(show_sets):
            d = P[(P["run_id"] == rid) & (P["eval_set"] == ev)]
            y, p = d["y"].values.astype(int), d["p"].values
            pT = sigmoid(logit(p) / temps[rid])
            ax = axes[i][j]
            ax.plot([0, 1], [0, 1], "k--", lw=0.8)
            for probs, lab in ((p, "before"), (pT, f"after (T={temps[rid]:.2f})")):
                bins = np.linspace(0, 1, 11)
                idx = np.digitize(probs, bins[1:-1])
                xs = [probs[idx == b].mean() for b in range(10) if (idx == b).sum() >= 5]
                ys = [y[idx == b].mean() for b in range(10) if (idx == b).sum() >= 5]
                ax.plot(xs, ys, marker="o", label=lab)
            ax.set_title(f"{meth} | {ev}")
            ax.set_xlabel("predicted P(biased)"); ax.set_ylabel("observed frequency")
            ax.legend()
    plt.tight_layout()
    plt.savefig(OUT / "reliability_final.png", dpi=150)
    plt.show()
else:
    print("\n(reliability plot skip: final-config runs predictions nahi mili)")

print("\nSaved in:", OUT)
print("""
KAISE PADHNA:
- In-domain test sets (mbib_*, babe_*) pe ECE after << before => temperature scaling kaam karta hai.
- gold_conf2 pe agar ECE kam hi sudhre => calibration domain-shift ke saath transfer nahi hoti (report me likho).
- Temperature poore val (mbib+babe mix, row dedupe ke bina) pe fit hui hai.
NEXT: 08c (dobara), phir NB11 (updated).
""")

In [ ]:
# ============================================================
# BIASCOPE — Notebook 08b: Explainability (Integrated Gradients) + Robustness
# CPU-friendly; GPU mile to bade samples. Results Drive me save hote hain.
# ============================================================
!pip -q uninstall -y torchao
!pip -q install -U peft transformers accelerate indic-transliteration

import ast, bisect, gc, html, json, math, random, re, string, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import roc_auc_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers.utils import logging as hf_logging
from peft import PeftModel, PeftConfig
from google.colab import drive

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()
drive.mount("/content/drive")

# ---------------- Config ----------------
RUN_IG, RUN_ROBUST = True, True
SEED = 42
MAX_LEN = 160
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    N_IG_BABE, N_IG_GOLD, N_ROB, IG_STEPS, BS = 300, 200, 1000, 32, 64
else:                       # CPU defaults
    N_IG_BABE, N_IG_GOLD, N_ROB, IG_STEPS, BS = 60, 80, 400, 20, 16
N_BOOT = 300
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

BASE_DIR = Path("/content/drive/MyDrive/BIASCOPE")
FINAL = BASE_DIR / "models" / "final" / "xlmr_lora_mbib_babe_s42"
FALLBACK = BASE_DIR / "models" / "nb07" / "xlmr_lora_en_hi_s42"
ADAPTER = FINAL if FINAL.exists() else FALLBACK
assert ADAPTER.exists(), f"Koi adapter nahi mila: {FINAL} ya {FALLBACK}"
TAG = ADAPTER.name
OUT_DIR = BASE_DIR / "outputs" / "nb08_explain_robust"
OUT_DIR.mkdir(parents=True, exist_ok=True)
BABE_HI = BASE_DIR / "data" / "multilingual" / "babe" / "babe_hi.csv"
GOLD_PATHS = [BASE_DIR / "data" / "multilingual" / "gold" / "hindi_gold_llm_labelled.csv",
              Path("/content/hindi_gold_llm_labelled.csv")]

print("=" * 70)
print("BIASCOPE — NOTEBOOK 08b | device:", DEVICE, "| adapter:", TAG)
if ADAPTER == FALLBACK:
    print("⚠ FALLBACK adapter (sirf MBIB pe trained) — BABE explainability ke liye weak model.")
    print("  07c ke baad (models/final/xlmr_lora_mbib_babe_s42) dobara chalao; results alag tag se save honge.")
print("=" * 70)

# ---------------- Load model ----------------
tok = AutoTokenizer.from_pretrained(str(ADAPTER))
assert tok.is_fast, "Fast tokenizer chahiye (offset mapping ke liye)"
base_name = PeftConfig.from_pretrained(str(ADAPTER)).base_model_name_or_path
base = AutoModelForSequenceClassification.from_pretrained(base_name, num_labels=2)
model = PeftModel.from_pretrained(base, str(ADAPTER)).to(DEVICE).eval()
inner = model.base_model.model if hasattr(model, "base_model") and hasattr(model.base_model, "model") else model
emb_layer = inner.get_input_embeddings()
print("✓ Model loaded | base:", base_name)

# ---------------- Data ----------------
from datasets import load_dataset
babe_test = load_dataset("mediabiasgroup/BABE")["test"].to_pandas().reset_index(drop=True)
babe_test["text"] = babe_test["text"].fillna("").astype(str)
babe_test["label"] = babe_test["label"].astype(int)

hi = pd.read_csv(BABE_HI, encoding="utf-8-sig")
hi = hi[(hi["split"] == "test") & hi["text_hi"].fillna("").astype(str).str.strip().ne("")].reset_index(drop=True)

gold_path = next((p for p in GOLD_PATHS if p.exists()), None)
gold = None
if gold_path is not None:
    g = pd.read_csv(gold_path, encoding="utf-8-sig")
    gold = g[(g["usable"] == 1) & (g["confidence"] >= 2)].reset_index(drop=True)
print(f"BABE en test: {len(babe_test)} | BABE hi test: {len(hi)} | gold_conf2: {0 if gold is None else len(gold)}")

# ---------------- Helpers ----------------
PUNCT = string.punctuation + "।॥“”‘’—–…·•"
def norm_word(w):
    return str(w).lower().strip(PUNCT)

def parse_words(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return set()
    if isinstance(x, (list, tuple, np.ndarray)):
        items = list(x)
    else:
        s = str(x).strip()
        try:
            items = ast.literal_eval(s) if s.startswith("[") else s.split(",")
        except Exception:
            items = s.strip("[]").split(",")
    return {norm_word(w) for it in items for w in str(it).split() if norm_word(w)}

@torch.no_grad()
def predict_probs(texts, bs=None):
    bs = bs or BS
    order = np.argsort([len(t) for t in texts])
    out = np.zeros(len(texts))
    for b in range(0, len(texts), bs):
        idx = order[b:b + bs]
        enc = tok([texts[i] for i in idx], return_tensors="pt", padding=True,
                  truncation=True, max_length=MAX_LEN).to(DEVICE)
        out[idx] = torch.softmax(model(**enc).logits.float(), -1)[:, 1].cpu().numpy()
    return out

def ig_tokens(text, steps=IG_STEPS):
    enc = tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN, return_offsets_mapping=True)
    offsets = enc.pop("offset_mapping")[0].tolist()
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    ids, mask = enc["input_ids"], enc["attention_mask"]
    with torch.no_grad():
        x = emb_layer(ids)
        base_ids = torch.full_like(ids, tok.pad_token_id)
        base_ids[0, 0], base_ids[0, -1] = tok.cls_token_id, tok.sep_token_id
        b = emb_layer(base_ids)
        fx = model(input_ids=ids, attention_mask=mask).logits[0]
        fb = model(inputs_embeds=b, attention_mask=mask).logits[0]
        delta = float((fx[1] - fx[0]) - (fb[1] - fb[0]))
        prob = float(torch.softmax(fx.float(), -1)[1])
    total = torch.zeros_like(x)
    for k in range(1, steps + 1):
        e = (b + (k / steps) * (x - b)).detach().requires_grad_(True)
        lg = model(inputs_embeds=e, attention_mask=mask).logits[0]
        g, = torch.autograd.grad(lg[1] - lg[0], e)
        total += g
    attr = ((x - b) * total / steps).sum(-1)[0].detach().cpu().numpy()
    return attr, offsets, delta, float(attr.sum()), prob

def word_scores(text, attr, offsets):
    spans = [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]
    starts = [s for s, _ in spans]
    scores = np.zeros(len(spans))
    for a, (s, e) in zip(attr, offsets):
        if e <= s:
            continue
        pos = max(s, e - 1)
        k = bisect.bisect_right(starts, pos) - 1
        if k >= 0 and pos < spans[k][1]:
            scores[k] += a
    return [text[s:e] for s, e in spans], scores

def boot_ci(vals, n=N_BOOT, seed=0):
    vals = np.asarray([v for v in vals if not np.isnan(v)], dtype=float)
    rng = np.random.default_rng(seed)
    means = [vals[rng.integers(0, len(vals), len(vals))].mean() for _ in range(n)]
    return float(vals.mean()), np.percentile(means, [2.5, 97.5])

# ============================================================
# A. INTEGRATED GRADIENTS vs reference words
# ============================================================
def eval_ig(items, n_max, name):
    """items: list of (id, text, reference_word_set)"""
    elig = []
    for id_, text, exp in items:
        norm = [norm_word(w) for w in text.split()]
        cand = [i for i, w in enumerate(norm) if w]
        m = sum(norm[i] in exp for i in cand)
        if 1 <= m < len(cand) and len(cand) >= 4:
            elig.append((id_, text, exp))
    print(f"\n[{name}] eligible: {len(elig)} / {len(items)}")
    random.Random(SEED).shuffle(elig)
    elig = elig[:n_max]
    rows, store, t0 = [], [], time.time()
    for j, (id_, text, exp) in enumerate(elig, 1):
        attr, offs, delta, sattr, prob = ig_tokens(text)
        words, sc = word_scores(text, attr, offs)
        norm = [norm_word(w) for w in words]
        cand = np.array([i for i, w in enumerate(norm) if w])
        is_exp = np.array([norm[i] in exp for i in cand])
        s = sc[cand]
        order = np.argsort(-s)
        n, m = len(cand), int(is_exp.sum())
        def rnd(k):
            kk = min(k, n)
            return 1 - math.comb(n - m, kk) / math.comb(n, kk) if n - m >= kk else 1.0
        rows.append(dict(set=name, id=id_, n_words=n, n_expert=m, prob=prob,
                         hit1=float(is_exp[order[0]]), hit3=float(is_exp[order[:3]].any()),
                         rand1=rnd(1), rand3=rnd(3),
                         word_auc=roc_auc_score(is_exp, s) if 0 < m < n else np.nan,
                         completeness_err=abs(sattr - delta) / (abs(delta) + 1e-6)))
        store.append((words, sc, [norm_word(w) in exp for w in words]))
        if j % 10 == 0 or j == len(elig):
            print(f"  {j}/{len(elig)} | elapsed {(time.time() - t0) / 60:.1f} min")
    return pd.DataFrame(rows), store

ig_frames, examples = [], []
if RUN_IG:
    items = [(r["uuid"] if "uuid" in babe_test.columns else i, r["text"], parse_words(r["biased_words"]))
             for i, r in babe_test.iterrows() if r["label"] == 1]
    d1, s1 = eval_ig(items, N_IG_BABE, "babe_en_vs_expert_words")
    ig_frames.append(d1); examples += [("BABE (en) — expert words underlined",) + t for t in s1[:15]]
    if gold is not None:
        gi = [(r["gold_id"], str(r["text_hi"]), parse_words(r["biased_span"]))
              for _, r in gold.iterrows() if r["bias_label"] == 1 and str(r["biased_span"]).strip()
              and str(r["biased_span"]) != "nan"]
        d2, s2 = eval_ig(gi, N_IG_GOLD, "gold_hi_vs_llm_span")
        ig_frames.append(d2); examples += [("Gold (hi) — LLM span underlined",) + t for t in s2[:15]]
    IG = pd.concat([f for f in ig_frames if len(f)])
    IG.to_csv(OUT_DIR / f"ig_results_{TAG}.csv", index=False)

    print("\n" + "=" * 70 + "\nINTEGRATED GRADIENTS — agreement with reference words\n" + "=" * 70)
    summ = []
    for name, d in IG.groupby("set"):
        h3, h3ci = boot_ci((d["hit3"] - d["rand3"]).values)
        a, aci = boot_ci(d["word_auc"].values)
        summ.append(dict(set=name, n=len(d), hit1=d["hit1"].mean(), rand1=d["rand1"].mean(),
                         hit3=d["hit3"].mean(), rand3=d["rand3"].mean(),
                         hit3_minus_rand=h3, ci_lo=h3ci[0], ci_hi=h3ci[1],
                         word_auc=a, auc_ci_lo=aci[0], auc_ci_hi=aci[1],
                         completeness_err_median=d["completeness_err"].median(),
                         prob_mean=d["prob"].mean()))
    S = pd.DataFrame(summ).set_index("set")
    display(S.round(3))
    print("hit3_minus_rand > 0 aur CI zero se door => IG random se behtar highlight karta hai.")
    print("word_auc 0.5 = random. completeness_err kam (<~0.2) = IG approximation theek.")
    S.to_csv(OUT_DIR / f"ig_summary_{TAG}.csv")

    def render(words, sc, is_exp):
        mx = max(float(np.max(sc)), 1e-9)
        parts = []
        for w, s, e in zip(words, sc, is_exp):
            a = max(float(s), 0) / mx
            style = f"background: rgba(255,80,80,{0.08 + 0.75 * a:.2f}); padding:1px 2px; border-radius:3px;"
            if e:
                style += " text-decoration: underline; font-weight: 600;"
            parts.append(f'<span style="{style}">{html.escape(w)}</span>')
        return " ".join(parts)
    body = "".join(f"<h4>{html.escape(t[0])}</h4><p style='font-size:17px;line-height:2'>{render(*t[1:])}</p>"
                   for t in examples)
    (OUT_DIR / f"ig_examples_{TAG}.html").write_text(
        f"<html><meta charset='utf-8'><body style='font-family:sans-serif;max-width:900px;margin:auto'>"
        f"<h3>IG word attributions (red = pushes towards 'biased')</h3>{body}</body></html>", encoding="utf-8")
    print("\nQualitative examples (top-5 IG words, * = reference word):")
    for t in examples[:6] + examples[15:21]:
        words, sc, is_exp = t[1], t[2], t[3]
        top = np.argsort(-sc)[:5]
        print(f"- [{t[0][:12]}] ", " | ".join(f"{words[i]}{'*' if is_exp[i] else ''}" for i in top))
    print("HTML saved:", OUT_DIR / f"ig_examples_{TAG}.html")

# ============================================================
# B. ROBUSTNESS
# ============================================================
def romanize(t):
    from indic_transliteration import sanscript
    r = sanscript.transliterate(t, sanscript.DEVANAGARI, sanscript.ITRANS).lower()
    return re.sub(r"[~^.]", "", r)

def typo(text, p, rng):
    out = []
    for w in text.split():
        if len(w) >= 4 and rng.random() < p:
            i = rng.randrange(1, len(w) - 2)
            w = w[:i] + w[i + 1] + w[i] + w[i + 2:]
        out.append(w)
    return " ".join(out)

def word_drop(text, p, rng):
    w = [x for x in text.split() if rng.random() > p]
    return " ".join(w) if w else text

HI_PUNCT_TBL = str.maketrans("", "", PUNCT)
COND = {
    "en": {"clean": lambda t, r: t,
           "typo5": lambda t, r: typo(t, 0.05, r),
           "typo15": lambda t, r: typo(t, 0.15, r),
           "word_drop10": lambda t, r: word_drop(t, 0.10, r),
           "lower_nopunct": lambda t, r: re.sub(r"[^\w\s]", "", t.lower())},
    "hi": {"clean": lambda t, r: t,
           "word_drop10": lambda t, r: word_drop(t, 0.10, r),
           "no_punct": lambda t, r: t.translate(HI_PUNCT_TBL),
           "romanized(hinglish-like)": lambda t, r: romanize(t)},
}

def boot_delta(y, p0, p1, n=N_BOOT, seed=0):
    rng = np.random.default_rng(seed)
    ds = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(set(y[i])) < 2:
            continue
        ds.append(roc_auc_score(y[i], p0[i]) - roc_auc_score(y[i], p1[i]))
    return np.percentile(ds, [2.5, 97.5])

if RUN_ROBUST:
    sets = {"babe_test_en": (babe_test["text"].tolist(), babe_test["label"].values, "en"),
            "babe_test_hi": (hi["text_hi"].astype(str).tolist(), hi["label"].astype(int).values, "hi")}
    if gold is not None:
        sets["gold_conf2"] = (gold["text_hi"].astype(str).tolist(), gold["bias_label"].astype(int).values, "hi")
    rows = []
    t0 = time.time()
    for sname, (texts, y, lang) in sets.items():
        idx = np.sort(np.random.default_rng(SEED).choice(len(texts), min(N_ROB, len(texts)), replace=False))
        texts_s, y_s = [texts[i] for i in idx], y[idx]
        clean_p = None
        for cname, fn in COND[lang].items():
            rr = random.Random(SEED)
            pert = [fn(t, rr) for t in texts_s]
            p = predict_probs(pert)
            if cname == "clean":
                clean_p = p
            auc = roc_auc_score(y_s, p)
            row = dict(set=sname, condition=cname, n=len(y_s), auc=auc,
                       f1=f1_score(y_s, (p >= 0.5).astype(int), zero_division=0),
                       flip_rate=float(((p >= 0.5) != (clean_p >= 0.5)).mean()), mean_p=float(p.mean()))
            if cname != "clean":
                lo, hi_ = boot_delta(y_s, clean_p, p)
                row.update(delta_auc=roc_auc_score(y_s, clean_p) - auc, d_lo=lo, d_hi=hi_)
            rows.append(row)
            print(f"  {sname:13s} {cname:26s} AUC={auc:.3f} | elapsed {(time.time() - t0) / 60:.1f} min")
    R = pd.DataFrame(rows)
    R.to_csv(OUT_DIR / f"robustness_{TAG}.csv", index=False)
    print("\n" + "=" * 70 + "\nROBUSTNESS (delta_auc > 0 = drop vs clean; CI = 95% paired bootstrap)\n" + "=" * 70)
    display(R.set_index(["set", "condition"]).round(3))
    fig, ax = plt.subplots(figsize=(11, 4.5))
    R.pivot(index="condition", columns="set", values="auc").plot(kind="bar", ax=ax)
    ax.axhline(0.5, color="gray", ls="--", lw=0.8)
    ax.set_ylabel("AUC"); ax.set_title(f"Robustness — {TAG}")
    plt.xticks(rotation=30, ha="right"); plt.tight_layout()
    plt.savefig(OUT_DIR / f"robustness_{TAG}.png", dpi=150); plt.show()

print("\nSaved in:", OUT_DIR)
print(f"""
REPORT ME YE LIKHNA:
- IG reference words: BABE me expert word-level annotation; Hindi gold me LLM-labelled spans (weak reference).
- Model: {TAG}. {'FALLBACK (MBIB-only) tha, final nahi.' if ADAPTER == FALLBACK else 'Final config (mbib+babe).'}
- 'romanized(hinglish-like)' deterministic transliteration hai (ITRANS lower-case), asli code-mixed Hinglish nahi.
- Samples chhote hain (IG {N_IG_BABE}/{N_IG_GOLD}, robustness {N_ROB}); CI wide ho sakti hai.
""")

In [ ]:
# ============================================================
# BIASCOPE — Notebook 08c: Final tables + significance tests (CPU)
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score
from google.colab import drive

drive.mount("/content/drive")
BASE_DIR = Path("/content/drive/MyDrive/BIASCOPE")
S07 = BASE_DIR / "outputs" / "nb07b_babe"
OUT = BASE_DIR / "outputs" / "nb08_final_tables"
OUT.mkdir(parents=True, exist_ok=True)
EVALS = ["mbib_en_test", "babe_test_en", "babe_test_hi", "gold_conf2"]
N_BOOT = 1000
rng = np.random.default_rng(0)

P = pd.concat([pd.read_csv(S07 / "predictions.csv"), pd.read_csv(S07 / "predictions_07c.csv")]) \
      .drop_duplicates(["run_id", "eval_set", "row_id"], keep="last")
R = pd.concat([pd.read_csv(S07 / "results.csv"), pd.read_csv(S07 / "results_07c.csv")]) \
      .drop_duplicates(["run_id", "eval_set", "group"], keep="last")
cal = pd.read_csv(BASE_DIR / "outputs" / "nb08_calibration" / "calibration_results.csv")

# ---------- 1. Domain gap (XLM-R, seed 42): kis data pe train -> kahan test ----------
print("=" * 70 + "\n1. DOMAIN GAP: AUC by training set (XLM-R, seed 42)\n" + "=" * 70)
dg = R[(R["group"] == "all") & (R["seed"] == 42) & (R["model"] == "xlmr") & R["eval_set"].isin(EVALS)]
dg_tab = dg.pivot_table(index=["method", "train_set"], columns="eval_set", values="auc")[EVALS].round(3)
display(dg_tab)
dg_tab.to_csv(OUT / "domain_gap_auc.csv")

# ---------- 2. LoRA vs Full: 3-seed ensemble + paired bootstrap ----------
def ens(method, ev, seeds=(42, 43, 44)):
    cols = []
    for s in seeds:
        d = P[(P["run_id"] == f"xlmr|{method}|mbib+babe|s{s}") & (P["eval_set"] == ev)].set_index("row_id")
        cols.append(d["p"])
    M = pd.concat(cols, axis=1, keys=list(seeds)).dropna()
    y = P[(P["run_id"] == f"xlmr|{method}|mbib+babe|s{seeds[0]}") & (P["eval_set"] == ev)] \
        .set_index("row_id")["y"].loc[M.index].astype(int)
    return M, y

print("\n" + "=" * 70 + "\n2. LoRA vs FULL FT (mbib+babe, 3-seed ensemble, paired bootstrap on AUC)\n" + "=" * 70)
rows = []
for ev in EVALS:
    Ml, yl = ens("lora", ev)
    Mf, yf = ens("full", ev)
    idx = Ml.index.intersection(Mf.index)
    y = yl.loc[idx].values
    pl, pf = Ml.loc[idx].mean(axis=1).values, Mf.loc[idx].mean(axis=1).values
    diffs = []
    for _ in range(N_BOOT):
        i = rng.integers(0, len(y), len(y))
        if len(set(y[i])) < 2:
            continue
        diffs.append(roc_auc_score(y[i], pl[i]) - roc_auc_score(y[i], pf[i]))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    single_l = np.mean([roc_auc_score(y, Ml.loc[idx, s].values) for s in Ml.columns])
    single_f = np.mean([roc_auc_score(y, Mf.loc[idx, s].values) for s in Mf.columns])
    rows.append(dict(eval_set=ev, n=len(y), auc_lora_ens=roc_auc_score(y, pl), auc_full_ens=roc_auc_score(y, pf),
                     lora_minus_full=roc_auc_score(y, pl) - roc_auc_score(y, pf), ci_lo=lo, ci_hi=hi,
                     significant=(lo > 0) or (hi < 0),
                     ens_gain_lora=roc_auc_score(y, pl) - single_l, ens_gain_full=roc_auc_score(y, pf) - single_f))
sig = pd.DataFrame(rows).set_index("eval_set").round(3)
display(sig)
print("significant=False => LoRA aur Full ka AUC gap noise ke andar hai (CI zero ko cover karta hai).")
sig.to_csv(OUT / "lora_vs_full_significance.csv")

# ---------- 3. Final report table ----------
print("\n" + "=" * 70 + "\n3. FINAL TABLE (mbib+babe): AUC mean±std over seeds | ECE raw → after temperature scaling\n" + "=" * 70)
fin = R[(R["train_set"] == "mbib+babe") & (R["group"] == "all") & R["eval_set"].isin(EVALS)]
fmt = lambda x: f"{x.mean():.3f}±{x.std():.3f}" if len(x) > 1 else f"{x.mean():.3f}"
auc_tab = fin.groupby(["model", "method", "eval_set"])["auc"].agg(fmt).unstack("eval_set")[EVALS]
auc_tab.columns = [f"AUC {c}" for c in auc_tab.columns]
seeds = fin.groupby(["model", "method"])["seed"].nunique().rename("seeds")
g = cal[cal["train_set"] == "mbib+babe"].groupby(["model", "method", "eval_set"])[["ece_before", "ece_after"]].mean()
g["cell"] = g.apply(lambda r: f"{r['ece_before']:.3f} → {r['ece_after']:.3f}", axis=1)
ece_tab = g["cell"].unstack("eval_set")[["babe_test_en", "gold_conf2"]]
ece_tab.columns = [f"ECE {c}" for c in ece_tab.columns]
final = pd.concat([seeds, auc_tab, ece_tab], axis=1)
display(final)
final.to_csv(OUT / "final_results_table.csv")
try:
    (OUT / "final_results_table.md").write_text(final.to_markdown(), encoding="utf-8")
except Exception as e:
    print("markdown export skip:", e)

print("\nSaved in:", OUT)
print("""
REPORT ME: LoRA vs Full ka claim 'significant' column ke hisaab se likho.
MuRIL sirf 1 seed hai: 'competitive' likho, 'better' nahi.
""")